
# Multiscale correlation-matrix analysis (LRG pipeline)

This notebook demonstrates how to take one or more fMRI correlation matrices, apply
band-aware noise filtering, and run a Laplacian renormalisation group (LRG) style
multiscale clustering workflow. Use it as a template for comparing spatial clusters
across frequency bands or imaging contrasts.



## Requirements and inputs
- A square correlation matrix for each contrast/band you want to analyse.
- (Optional) Raw time-series data if you want to recreate the correlation matrix from scratch using bandpass filters.
- The utilities in `multifunbrain.analysis` and `multifunbrain.core` for Laplacian diffusion, clustering, and filtering.

All paths are relative to the repository root so the notebook remains portable.


In [ ]:

from pathlib import Path
from typing import Iterable, List, Sequence

import networkx as nx
import numpy as np
from scipy.cluster.hierarchy import fcluster
from scipy.special import comb

from multifunbrain.analysis.corrnet import compute_correlation_matrix
from multifunbrain.analysis.lrglib import (
    compute_normalized_linkage,
    compute_optimal_threshold,
    graph_laplacian_and_spectrum,
    rho_matrix,
    symmetrized_inverse_distance,
)
from multifunbrain.core import band_filter, marchenko_pastur_density



## Helper functions
The helpers below keep the workflow modular: loading matrices, band-filtering
(optional), diffusion-based clustering, and partition agreement between matrices.


In [ ]:

def load_correlation_matrix(path: Path) -> np.ndarray:
    """Load a correlation matrix from `.npy`, `.npz`, `.csv`, or `.txt` files."""

    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix in {".npy"}:
        return np.load(path)
    if path.suffix == ".npz":
        with np.load(path) as data:
            return data[list(data.keys())[0]]
    if path.suffix in {".csv", ".txt"}:
        return np.loadtxt(path, delimiter="," if path.suffix == ".csv" else None)

    raise ValueError(f"Unsupported file type: {path.suffix}")


def prepare_correlation_matrix(matrix: np.ndarray, zero_diagonal: bool = True, clip: bool = True) -> np.ndarray:
    """Symmetrise and clean a correlation matrix before graph analysis."""

    corr = np.asarray(matrix, dtype=float)
    if corr.ndim != 2 or corr.shape[0] != corr.shape[1]:
        raise ValueError("Correlation matrix must be square.")

    corr = 0.5 * (corr + corr.T)
    if clip:
        corr = np.clip(corr, -1.0, 1.0)
    if zero_diagonal:
        np.fill_diagonal(corr, 0.0)
    return corr


def marchenko_pastur_denoise(corr: np.ndarray, gamma: float = 0.5, sigma: float = 1.0) -> np.ndarray:
    """Simple eigenvalue shrinkage using the Marchenko-Pastur support.

    Eigenvalues outside the theoretical support are set to the nearest boundary
    before reconstructing the matrix. Adjust `gamma` to reflect ``p/n`` where
    ``p`` is the number of regions and ``n`` the number of samples.
    """

    corr = prepare_correlation_matrix(corr)
    evals, evecs = np.linalg.eigh(corr)
    density = marchenko_pastur_density(evals, gamma=gamma, sigma=sigma)

    lam = np.asarray(evals)
    mask = density > 0
    if not np.any(mask):
        return corr

    lambda_min = lam[mask].min()
    lambda_max = lam[mask].max()
    lam_clipped = np.clip(lam, lambda_min, lambda_max)
    denoised = (evecs * lam_clipped) @ evecs.T
    return prepare_correlation_matrix(denoised)


def hierarchical_partitions_from_corr(
    corr: np.ndarray,
    tau_values: Sequence[float],
    edge_threshold: float = 0.0,
) -> List[dict]:
    """Run the diffusion/LRG-style clustering pipeline for a correlation matrix."""

    cleaned = prepare_correlation_matrix(corr)
    if edge_threshold > 0:
        cleaned = np.where(cleaned >= edge_threshold, cleaned, 0.0)

    graph = nx.from_numpy_array(cleaned)
    L, spectrum = graph_laplacian_and_spectrum(graph, weight="weight", normalized=True)

    partitions = []
    for tau in tau_values:
        dists = symmetrized_inverse_distance(tau, lambda t: rho_matrix(t, L))
        linkage_matrix, labels, tmax = compute_normalized_linkage(dists, graph, labelList="numbers")
        flat_threshold, _, _, _ = compute_optimal_threshold(linkage_matrix)
        partition = fcluster(linkage_matrix, flat_threshold, criterion="distance")
        partitions.append(
            {
                "tau": float(tau),
                "partition": partition,
                "linkage_matrix": linkage_matrix,
                "linkage_labels": labels,
                "tmax": tmax,
                "flat_threshold": flat_threshold,
            }
        )
    return partitions


def adjusted_rand_index(labels_a: Iterable[int], labels_b: Iterable[int]) -> float:
    """Compute the Adjusted Rand Index without external dependencies."""

    labels_a = np.asarray(labels_a)
    labels_b = np.asarray(labels_b)
    if labels_a.shape != labels_b.shape:
        raise ValueError("Label arrays must have the same shape.")

    n = labels_a.size
    if n == 0:
        return 0.0

    contingency = np.histogram2d(labels_a, labels_b, bins=(labels_a.max() + 1, labels_b.max() + 1))[0]
    sum_comb_c = comb(contingency.sum(axis=1), 2).sum()
    sum_comb_k = comb(contingency.sum(axis=0), 2).sum()
    sum_comb = comb(contingency, 2).sum()
    total_pairs = comb(n, 2)

    expected_index = (sum_comb_c * sum_comb_k) / total_pairs if total_pairs else 0.0
    max_index = (sum_comb_c + sum_comb_k) / 2
    if max_index == expected_index:
        return 1.0

    return float((sum_comb - expected_index) / (max_index - expected_index))


def compare_partition_sets(set_a: List[dict], set_b: List[dict]) -> List[dict]:
    """Compare two sets of partitions across all tau pairs using ARI."""

    comparisons: List[dict] = []
    for part_a in set_a:
        for part_b in set_b:
            ari = adjusted_rand_index(part_a["partition"], part_b["partition"])
            comparisons.append(
                {
                    "tau_a": part_a["tau"],
                    "tau_b": part_b["tau"],
                    "ari": ari,
                }
            )
    return comparisons



## Example data
Below we synthesise two band-limited correlation matrices from toy signals. Replace
`corr_band_a` and `corr_band_b` with your own matrices to reuse the pipeline.


In [ ]:

rng = np.random.default_rng(42)

n_regions = 32
n_timepoints = 600
fs = 1 / 1.2

# Base multichannel signal with shared latent trends
base_signal = rng.normal(size=(n_regions, n_timepoints))
trend = np.sin(np.linspace(0, 6 * np.pi, n_timepoints))
base_signal += 0.2 * trend

# Create two contrasted bands
slow5 = band_filter(base_signal, low=0.01, high=0.027, fs=fs)
slow4 = band_filter(base_signal, low=0.027, high=0.073, fs=fs)

corr_band_a = compute_correlation_matrix(slow5)
corr_band_b = compute_correlation_matrix(slow4)

# Optional denoising step
corr_band_a = marchenko_pastur_denoise(corr_band_a, gamma=0.6)
corr_band_b = marchenko_pastur_denoise(corr_band_b, gamma=0.6)



## Run diffusion/LRG clustering
Choose the diffusion scales (`tau_values`) to explore the multiscale structure of
the correlation networks. The partitions keep track of dendrogram thresholds and
labels for downstream comparison.


In [ ]:

tau_values = np.logspace(-2, 1, 6)

partitions_a = hierarchical_partitions_from_corr(corr_band_a, tau_values, edge_threshold=0.0)
partitions_b = hierarchical_partitions_from_corr(corr_band_b, tau_values, edge_threshold=0.0)

for part in partitions_a[:2]:
    print(f"τ={part['tau']:.3f} -> {len(np.unique(part['partition']))} clusters (band A)")
for part in partitions_b[:2]:
    print(f"τ={part['tau']:.3f} -> {len(np.unique(part['partition']))} clusters (band B)")



## Cross-matrix clustering agreement
We use the Adjusted Rand Index (ARI) to quantify how similar the community
assignments are between bands or contrasts at different diffusion scales.


In [ ]:

comparison = compare_partition_sets(partitions_a, partitions_b)

for row in comparison:
    if row["tau_a"] == row["tau_b"]:
        print(f"τ={row['tau_a']:.3f} | ARI={row['ari']:.3f}")



## Next steps
- Replace the synthetic matrices with your real band/contrast correlation matrices.
- Tune the diffusion scales to match the temporal resolution of your experiment.
- Persist `partitions_*` to disk (e.g., via `pickle` or JSON) to compare many
  subjects or contrasts outside the notebook.
